In order to successfully complete this assignment, you must follow all the instructions in this notebook and upload your edited ipynb file to [D2L](http://d2l.msu.edu/) with your answers on or before **Friday, April 7th at 11:59pm ET**.

**BIG HINT:** Read the entire homework before starting. 

# Homework 4: Projection and diagonalization

In [1]:
%matplotlib inline
import matplotlib.pylab as plt
import numpy as np
import sympy as sym
import math
sym.init_printing(use_unicode=True)

In [2]:
# import for checking answers
import os.path
from urllib.request import urlretrieve
if not os.path.isfile("answercheck.py"):
    urlretrieve('https://raw.githubusercontent.com/colbrydi/jupytercheck/master/answercheck.py', 'answercheck.py');
from answercheck import checkanswer

# 1. Computing the exponential of a matrix (35 pts)

In this exercise, we will learn how to apply the exponential function $e^x$ to a matrix in a way that makes sense.

The first step is to use a [fact from calculus](https://www.mathsisfun.com/algebra/taylor-series.html) that we can represent the function $e^x$ by its *Taylor series*, that is, a polynomial with infinitely many terms:

$$e^x = \frac{x^0}{0!} + \frac{x^1}{1!} + \frac{x^2}{2!} + \frac{x^3}{3!} + \ldots + \frac{x^n}{n!} + \ldots$$

The following function computes the value of this Taylor series where we only use `d` many terms, i.e., use a Taylor polynomial of degree `d`.

In [25]:
def exp_taylor(x, d):
    y = 0.0
    for n in range(d + 1):
        y += (1 / math.factorial(n)) * x ** n 
    return y

&#9989;  **<font color=red>QUESTION 1.1:</font>** (3 pts) Find the smallest degree Taylor polynomial that can compute the value of $e^1 = e$ with enough accuracy that `numpy` cannot distinguish it from `np.e` using `np.isclose`.

*(Hint: try using a for loop.)*

In [35]:
for d in range(1, 100):
    if np.isclose(exp_taylor(1, d), np.e):
        print("The smallest degree Taylor polynomial that computes e^1 to the same value as np.e is", d)
        break

The smallest degree Taylor polynomial that computes e^1 to the same value as np.e is 8


The nice thing about polynomials is that they only use two operations: multiplication and addition. We can do both of these things to matrices! So even though we don't really have a way to compute $e^X$ when $X$ is a matrix, we can compute

$$\frac{X^0}{0!} + \frac{X^1}{1!} + \frac{X^2}{2!} + \frac{X^3}{3!} + \ldots + \frac{X^n}{n!} + \ldots$$

(or at least an approximation to it using a finite number of terms). This is the definition of the *matrix exponential* $e^X$.

&#9989;  **<font color=red>QUESTION 1.2:</font>** (5 pts) Using the function [`np.linalg.matrix_power`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_power.html), modify the function above to compute the Taylor polynomial of degree `d` using a matrix `X`.

In [44]:
# Put your code here

def exp_taylor_matrix(X, d):
    y = 0.0
    for n in range(d + 1):
        y += (1 / math.factorial(n)) * np.linalg.matrix_power(X, n) # MODIFY THIS LINE
    return y

Test your code below:

In [45]:
X = np.matrix([[2, 3, -1], [1, -6, 2], [2, 1, -3]])
Y = exp_taylor_matrix(X, 1000) # We use a very high degree to get a good approximation

from answercheck import checkanswer
checkanswer.matrix(Y,"ebc8b8e6828a967e5ee6634bf3f003eb", decimal_accuracy=3);

Testing [[ 8.965  3.2   -0.497]
 [ 1.992  0.716 -0.074]
 [ 3.771  1.352 -0.143]]
Answer seems to be correct



&#9989;  **<font color=red>QUESTION 1.3:</font>** (5 pts) Compare your result to applying `numpy`'s exponential function [`np.exp`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html) to the matrix $X$. Why are the results not equal?

In [51]:
# Put your code for the above question here 
np.exp(X)

matrix([[7.38905610e+00, 2.00855369e+01, 3.67879441e-01],
        [2.71828183e+00, 2.47875218e-03, 7.38905610e+00],
        [7.38905610e+00, 2.71828183e+00, 4.97870684e-02]])

The results are not equal because the matrix exponential function and the element-wise exponential function act differently on matrices. The matrix exponential function takes the sum of the Taylor series terms, where each term is the matrix raised to a power and divided by a factorial. On the other hand, the element-wise exponential function computes the exponential of each individual element in the matrix. So while they have similar properties, they are fundamentally different operations.

Great! Now we have a way to take the exponential of a matrix. However, as we learned, matrix multiplication is $O(n^3)$ for an $n \times n$ matrix, and this gets expensive when you're doing it over and over to compute the powers of a matrix.

But section 3 of ICA 15 gave us another way to compute powers of matrices. We will use this to create a faster `exp_taylor_matrix` function.

For any diagonalizable matrix $X$, we can find two matrices $C$ and $D$ where $D$ is diagonal such that

$$X = CDC^{-1}.$$

&#9989;  **<font color=red>QUESTION 1.4:</font>** (5 pts) Explain in words what $C$ and $D$ are and how you would compute them in `numpy`.

Matrix $C$ is a matrix whose columns are the eigenvectors of the matrix $X$ and matrix $D$ is a diagonal matrix whose diagonal elements are the eigenvalues of the matrix $X$. In other words, we can factorize a diagonalizable matrix $X$ as $X = CDC^{-1}$, where $C$ is the matrix of eigenvectors and $D$ is the matrix of eigenvalues.

In numpy, we can compute the eigenvectors and eigenvalues of a matrix using numpy.linalg.eig function. We can then construct the matrix $C$ using the eigenvectors, and the matrix $D$ using the eigenvalues. We can then compute $X = CDC^{-1}$ to get the diagonalizable matrix $X$.

By using the fact that $I = C^{-1}C$, repeated multiplication gives

$$
\begin{aligned}
X^n 
    &= (CDC^{-1})^n \\
    &= CDC^{-1} \times CDC^{-1} \times CDC^{-1} \\
    &= CD(C^{-1}C)D(C^{-1}C) \cdots (C^{-1}C)DC^{-1} \\
    &= CDIDI \cdots IDC^{-1} \\
    &= CD^nC^{-1}.
\end{aligned}
$$

&#9989;  **<font color=red>QUESTION 1.5:</font>** (6 pts)

Define the polynomial (of matrices)

$$p(X) = I + X + 2 X^2 + 3 X^3.$$

Using the distributive property of matrix multiplication and the power property above, show that if $X$ is diagonalizable, then

$$ p(X) = C \times p(D) \times C^{-1} .$$

Now, if $X$ is diagonalizable then, there exists a diagonal matrix $D$ and a matrix $C$ such that $X=CDC^{-1}$. Then we have

\\begin{align*}
p(X) &= I + X + 2X^2 + 3X^3 \\
&= I + CDC^{-1} + 2(CDC^{-1})^2 + 3(CDC^{-1})^3 \\
&= I + CDC^{-1} + 2CDC^{-1}CDC^{-1} + 3CDC^{-1}CDC^{-1}CDC^{-1} \\
&= CIC^{-1} + CD^1C^{-1} + 2CD^2C^{-1} + 3CD^3C^{-1} \\
&= C(I + D^1 + 2D^2 + 3D^3)C^{-1} \\
&= C * p(D) * C^{-1}.
\end{align*}

Therefore, we have shown that $ p(X) = C * p(D) * C^{-1}$ if $X$ is diagonalizable.

&#9989;  **<font color=red>QUESTION 1.6:</font>** (6 pts) For the diagonal matrix
$D = 
\begin{bmatrix}
    \lambda_1 & 0 \\
    0 & \lambda_2
\end{bmatrix}
$, using the fact that
$D^n = 
\begin{bmatrix}
    \lambda_1^n & 0 \\
    0 & \lambda_2^n
\end{bmatrix},
$

show that

$$p(D) = \begin{bmatrix}
            p(\lambda_1) & 0 \\
            0 & p(\lambda_2)
         \end{bmatrix}$$
         
(where $p$ works on numbers instead of matrices by replacing the identity $I$ with $1$).

$p(D) = 1 + D + 2D^2 + 3D^3$

Using the distributive property of matrix multiplication, we can write

$\begin{matrix}
\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} &+\begin{bmatrix} \lambda_1 & 0 \\ 0 & \lambda_2 \end{bmatrix}&+ 2\begin{bmatrix} \lambda_1^2 & 0 \\ 0 & \lambda_2^2 \end{bmatrix}&+3\begin{bmatrix} \lambda_1^3 & 0 \\ 0 & \lambda_2^3 \end{bmatrix}
\end{matrix}$


$\begin{matrix} = \begin{bmatrix} 1 + \lambda_1 + 2\lambda_1^2 + 3\lambda_1^3 & 0 \\ 0 & 1 + \lambda_2 + 2\lambda_2^2 + 3\lambda_2^3 \end{bmatrix}\end{matrix}$

$\begin{matrix} = \begin{bmatrix} p(\lambda_1) & 0 \\ 0 & p(\lambda_2)\end{bmatrix}\end{matrix}$



It turns out that the same thing is true for the Taylor series of $e^x$. That is, if 

$$p(X) = \frac{X^0}{0!} + \frac{X^1}{1!} + \frac{X^2}{2!} + \frac{X^3}{3!} + \ldots + \frac{X^n}{n!} + \ldots $$

then

$$p(X) = C \times p(D) \times C^{-1} 
= C
\begin{bmatrix}
p(\lambda_1) & 0 & 0 & 0 \\
0 & p(\lambda_2) & 0 & 0\\
0 & 0 & \ddots & 0 \\
0 & 0 & 0 & p(\lambda_n)
\end{bmatrix}
C^{-1}$$
But if we remember that the Taylor series is equal to the exponential function, that is, $p(\lambda_i) = e^{\lambda_i}$, we have
$$e^X = p(X) 
= C
\begin{bmatrix}
e^{\lambda_1} & 0 & 0 & 0 \\
0 & e^{\lambda_2} & 0 & 0\\
0 & 0 & \ddots & 0 \\
0 & 0 & 0 & e^{\lambda_n}
\end{bmatrix} C^{-1}.
 $$
 
So let's put this all together to write a faster `exp_taylor_matrix` function.

&#9989;  **<font color=red>QUESTION 1.7:</font>** (5 pts) Fill in the code in the following function. Add any lines where you need to.

In [52]:
def faster_exp_taylor_matrix(X):
    eigenvals = np.linalg.eig(X)[0]# FIND EIGENVALUES
    expD = np.diag(np.exp(eigenvals))
    C = np.linalg.eig(X)[1]# FIND C
    Y = C @ expD @ np.linalg.inv(C)
    return Y

Rerun the test from before to check your answer:

In [53]:
X = np.matrix([[2, 3, -1], [1, -6, 2], [2, 1, -3]])
Y = faster_exp_taylor_matrix(X)


from answercheck import checkanswer
checkanswer.matrix(Y,"ebc8b8e6828a967e5ee6634bf3f003eb", decimal_accuracy=3);

Testing [[ 8.965  3.2   -0.497]
 [ 1.992  0.716 -0.074]
 [ 3.771  1.352 -0.143]]
Answer seems to be correct



Notice that the faster function is indeed faster! For larger matrices and higher degree polynomials, this would make even more of a difference.

In [321]:
# Run this cell!

import time
t_start = time.time()
Y = exp_taylor_matrix(X, 100)
print("Original function time:", time.time() - t_start, "seconds")
t_start = time.time()
Y = faster_exp_taylor_matrix(X)
print("Faster function time:", time.time() - t_start, "seconds")

Original function time: 0.011111974716186523 seconds
Faster function time: 0.0013248920440673828 seconds


# 2. Projection (35 pts)

Given the vector $v=[-2,1,3]^\top$ and the plane $W = \{[x_1,x_2,x_3]^\top: x_1+x_2-x_3=0\},$ in
$\mathbb{R}^3$, we want to find the projection of $v$ onto $W$ using the following steps.



&#9989;  **<font color=red>QUESTION 2.1:</font>** (5 pts) 
Use the equation defining the plane to solve for $x_1$ in terms of $x_2$ and $x_3$. Use this to write the vectors in $W$ depending on just the values $x_2$ and $x_3$.

(*Hint: This is very similar to solving for the nullspace of a matrix where you have two free variables.*)

The equation defining the plane $W$ is $x_1 + x_2 - x_3 = 0$. We can solve for $x_1$ in terms of $x_2$ and $x_3$ by rearranging the equation as follows:

$x_1 = - x_2 + x_3$

Now, we can write any vector in $W$ as:

$\begin{matrix}  \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} - x_2 + x_3 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} 1 \\ 0 \\ 1 \end{bmatrix} or \begin{bmatrix} -1 \\ 1 \\ 0 \end{bmatrix} ... \end{matrix}$


&#9989;  **<font color=red>QUESTION 2.2:</font>** (5 pts)
Rewrite this your answer to QUESTION 2.1 to be a linear combination of the form $x_2 v_1 + x_3 v_2$, where $v_1$ and $v_2$ are vectors.

Check your values of $v_1$ and $v_2$ below:

In [78]:
# Put your answer for the above question here
v1 = np.matrix([[-1],[1],[0]])
v2 = np.matrix([[1],[0],[1]])

In [79]:
from answercheck import checkanswer
checkanswer.vector(v1 / np.linalg.norm(v1),"2a0160bf571ec2047b7cb0355066aa24");
checkanswer.vector(v2 / np.linalg.norm(v2),"0d5dd28ffe9384507c3378da18db6869");


    Trying to convert to a column vector using ```A = A.T```.


    Converting to positive values of zero using  ```A[A==-0] = 0```.

Testing [[-0.70711  0.70711  0.     ]]
Answer seems to be correct


    Trying to convert to a column vector using ```A = A.T```.


    Converting to positive values of zero using  ```A[A==-0] = 0```.

Testing [[0.70711 0.      0.70711]]
Answer seems to be correct



&#9989;  **<font color=red>QUESTION 2.3:</font>** (5 pts)
Check that `v1` and `v2` are linearly independent. You may either do this by hand or using Python.

To check if $v_1$ and $v_2$ are linearly independent, we can create a matrix with these vectors as its columns and compute its rank. If the rank is 2, then the vectors are linearly independent, otherwise they are linearly dependent.

In [80]:
# Use this cell for an answer with Python (leave blank if doing by hand)
V = np.matrix([[-1, 1], [1,0],[0,1]])
np.linalg.matrix_rank(V)

2

Since rank and number of columns is equal, they're linearly independent

This tells us that these two vectors are a basis for $W$, but you can check that they are not orthonormal.

&#9989;  **<font color=red>QUESTION 2.4:</font>** (5 pts)
Since an orthonormal basis is easier to work with, orthogonalize `v1` and `v2` by applying the Gram-Schmidt process. Store the resulting orthonormal vectors as `w1` and `w2`

In [243]:
# Put your answer to the above question here
v1 = np.matrix([-1,1,0])
v2 = np.matrix([1,0,1])

w1 = v1/np.linalg.norm(v1)
u2 = v2 - (np.dot(v2,w1.T)/np.dot(w1,w1.T))*w1
w2 = u2/np.linalg.norm(u2)

print(w1)
print(w2)


[[-0.70710678  0.70710678  0.        ]]
[[0.40824829 0.40824829 0.81649658]]


&#9989;  **<font color=red>QUESTION 2.5:</font>** (5 pts) Use python to verify that {`w1`,`w2`} is indeed an orthonormal basis of the plane $W$. Specifically, verify

-  each basis vector has unit length 
-  orthogonality between different basis vectors.
-  `w1`, `w2` $\in W$ 

In [249]:
# Put your answer to the above question here

# To check for unit length
print(round(np.linalg.norm(w1)), 'should be 1')
print(round(np.linalg.norm(w2)), 'should be 1' )

# To check for orthogonality between different basis vectors.
print((np.dot(w1, w2.T)[0,0]) , 'should return 0')

# w1, w2  ∈𝑊
print(w1[0,0] + w1[0,1] - w1[0,2], 'should return 0')
print(w2[0,0] + w2[0,1] - w2[0,2], 'should return 0')


1 should be 1
1 should be 1
0.0 should return 0
0.0 should return 0
0.0 should return 0


Recall that in Pre-class 13, the following mathematical definition of projection onto a subspace is defined.

**Definition**: Let $W$ be a subspace of $R^n$ of dimension $m$. Let $\{w_1,\cdots,w_m\}$ be an orthonormal basis for $W$. Then the projection of vector $v$ in $R^n$ onto $W$ is denoted as $\mbox{proj}_Wv$ and is defined as 
$$\mbox{proj}_Wv = (v\cdot w_1)w_1+(v\cdot w_2)w_2+\cdots+(v\cdot w_m)w_m$$


&#9989;  **<font color=red>QUESTION 2.6:</font>** (5 pts) Use this formula to find $P_w v$, the projection of $v=[-2,1,3]^T$ onto $W$


In [272]:
# Put your answer to the above question here
v = np.matrix([[-2,1,3]])
Pwv = (np.dot(v,w1.T)* w1) + (np.dot(v,w2.T)* w2)

In [273]:
from answercheck import checkanswer
checkanswer.vector(Pwv,"02b1f7fc612d9c4721e21e45f4444688");

Testing [[-0.66667  2.33333  1.66667]]
Answer seems to be correct



&#9989;  **<font color=red>QUESTION 2.7:</font>** (5 pts) What is the geometric meaning of $v-P_W v$? Use this to find the distance from $v$ to $W$.

The vector  $v-P_W v$ is the orthogonal component of ${v}$ with respect to the subspace $W$. Geometrically, it is the shortest vector connecting ${v}$ to $W$, which means it is the distance from ${v}$ to $W$. Therefore, to find the distance from ${v}$ to $W$, we just need to compute the length of ${v}-{P_W v}$.



In [278]:
# Put your code to find the distance here
distance = np.linalg.norm(v -Pwv)

In [279]:
from answercheck import checkanswer
checkanswer.matrix(distance,"dd5b8b47bdfbbad36cedf5ee17fe60b1");


    Trying to convert to a array matrix using ```A = np.matrix(A)```.

Testing [[2.3094]]
Answer seems to be correct



# 3. Properties of orthogonal matrices (30 pts)


In pre-class 13, we learned the definition of orthogonal matrices. Let us  review it here.

**Definition:** A $n\times n$ matrix $U$ is orthogonal if the columns of $U$ form an orthonormal basis of  $\mathbb {R} ^{n}$.

Orthogonal matrices have many alternative definitions. Explictly, the following conditions are all equivalent.

1. $U$ is orthogonal

2. the rows of $U$ form an orthonormal basis of $\mathbb{R}^{n}$.

3. the columns of $U$ form an orthonormal basis of $\mathbb{R}^{n}$.

4. U is invertible, and $U^T = U^{-1}$.

5. For any two vectors $x$ and $y$ in $\mathbb{R}^n$, multiplication by $U$ preserves their dot product; that is, $(Ux)\cdot (Uy) = x \cdot y$.

&#9989;  **<font color=red>QUESTION 3.1:</font>** (5 pts) Let $v_1 = \left[\frac{1}{\sqrt 2}, 0, \frac{1}{\sqrt 2}\right]$, $v_2 = [0, 1, 0]$, $v_3 = \left[\frac{1}{\sqrt 2}, 0, -\frac{1}{\sqrt 2}\right]$.

Use one of conditions 2, 3, or 4 above to verify that $v_1,v_2,v_3$ form an orthonormal basis of $\mathbb{R}^3$.

We can verify that $v_1, v_2, v_3$ form an orthonormal basis of $\mathbb{R}^3$ by checking that the columns of the matrix $U$ formed by stacking $v_1, v_2, v_3$ are orthonormal.



In [283]:
# Put your answer for the above question here
v1 = np.array([1/np.sqrt(2), 0, 1/np.sqrt(2)])
v2 = np.array([0, 1, 0])
v3 = np.array([1/np.sqrt(2), 0, -1/np.sqrt(2)])

U = np.column_stack((v1, v2, v3))
print(np.allclose(np.dot(U.T, U), np.identity(3)))


True


&#9989;  **<font color=red>QUESTION 3.2:</font>** (5 pts) Use python code  to construct the following two $3\times 3$ transformation matrices: 
+ Rotation matrix `R` that rotates a given vector in $\mathbb{R}^3$ around the $y$-axis counterclockwise by an angle of $40^\circ$ 
+ Reflection matrix `F` that reflects a given vector in $\mathbb{R}^3$ with respect to the $x$-$y$ plane.

In [287]:
# Put your answer for the above question here
theta = 40 * np.pi / 180
R = np.matrix([[np.cos(theta), 0, np.sin(theta)],
               [0,1,0],
               [-np.sin(theta),0,np.cos(theta)]])
F = np.array([[1, 0, 0],
              [0, 1, 0],
              [0, 0, -1]])

&#9989;  **<font color=red>QUESTION 3.3:</font>** (5 pts) Verify that `R` and `F` are orthogonal matrices by computing their inverses and verifying that $R^{-1}= R^T$ and $F^{-1}= F^T$. It is also a fact that the product of orthogonal matrices is also an orthogonal matrix. Verify that the product `RF` is an orthogonal matrix.

In [294]:
# Put your answer for the above question here
# Verify that R is orthogonal
R_inv = np.linalg.inv(R)
R_transpose = (R).T
print(np.allclose(R_inv, R_transpose))

# Verify that F is orthogonal
F_inv = np.linalg.inv(F)
F_transpose = (F).T
print(np.allclose(F_inv, F_transpose))

# Verify that RF is orthogonal
RF = R*F
RF_inv = np.linalg.inv(RF)
RF_transpose = (RF).T
print(np.allclose(RF_inv, RF_transpose))


True
True
True


&#9989;  **<font color=red>QUESTION 3.4:</font>** (5 pts) The determinant of the orthogonal matrix will always be +1 or -1. Verify that the determinant of`R`, `F` and `RF` is either +1 or -1.

In [300]:
# Put your answer for the above question here
print('Determinant of R =', round(np.linalg.det(R)))
print('Determinant of F =', round(np.linalg.det(F)))
print('Determinant of RF =', round(np.linalg.det(RF)))

Determinant of R = 1
Determinant of F = -1
Determinant of RF = -1


&#9989;  **<font color=red>QUESTION 3.5:</font>** (5 pts) Compute the eigenvalues of `R`, `F` and `RF`. What did you find about the magnitudes of these eigenvalues? Could you come up with a theoretical explanation for this phenomenon?

In [311]:
# Put your answer for the above question here
R_eig = np.linalg.eig(R)[0]
F_eig = np.linalg.eig(F)[0]
RF_eig = np.linalg.eig(RF)[0]
print('EigVals of R:', np.abs(R_eig))
print('EigVals of F:', np.abs(F_eig))
print('EigVals of RF:', np.abs(RF_eig))

EigVals of R: [1. 1. 1.]
EigVals of F: [1. 1. 1.]
EigVals of RF: [1. 1. 1.]


All their absolute values are 1. Thats because rotation and reflection matrices do not strech and eigenvalues essentially tell us about the strech of eigenvectors. 

&#9989;  **<font color=red>QUESTION 3.6:</font>** (5 pts) Are each of the following statements true or false? 

1.  The identity matrix is orthogonal.   
2.  Every diagonal matrix is orthogonal. 
3.  If $A$ is an $n \times n$ orthogonal matrix, and $x$ is any column vector in $\mathbb{R}^n$, then $\|Ax\| = \|x\|$.  
4.  Every entry of an orthogonal matrix must be between $0$ and $1$ inclusive.  
5.  An orthogonal matrix must be symmetric.  

- True
- False
- True
- False
- False

### Congratulations, we're done!